# Week 25, Day 5: Controls: what else could make rotations?

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w025/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.load(dataset("churchland2012_reach.npz"))
rates, times = data["rates"].astype(float), data["times"]
C, T, N = rates.shape
Xn = rates / (rates.max(axis=(0, 1)) - rates.min(axis=(0, 1)) + 5)
X = Xn - Xn.mean(axis=0, keepdims=True)
win = (times >= -50) & (times <= 150); tw = times[win]
Xw = X[:, win, :]
dt = 0.01
basis = []
for i in range(6):
    for j in range(i + 1, 6):
        E = np.zeros((6, 6)); E[i, j], E[j, i] = 1, -1; basis.append(E)

def rotation_r2(A):
    """PCA to six components, then the R squared of the best skew-symmetric fit of dZ/dt = MZ."""
    D = A.reshape(-1, A.shape[-1]); mu = D.mean(0)
    Vt = np.linalg.svd(D - mu, full_matrices=False)[2]
    Z = (A - mu) @ Vt[:6].T
    Zm, dZ = Z[:, :-1].reshape(-1, 6), (np.diff(Z, axis=1) / dt).reshape(-1, 6)
    G = np.stack([(Zm @ E.T).ravel() for E in basis], axis=1)
    h = np.linalg.lstsq(G, dZ.ravel(), rcond=None)[0]
    return 1 - np.sum((dZ.ravel() - G @ h) ** 2) / np.sum((dZ - dZ.mean(0)) ** 2)

real = rotation_r2(Xw)
print(f"real data: rotational R2 = {real:.3f}")

### Exercise: A condition-shuffle null

Using a local rng (seed 0), make 20 shuffled copies of Xw, permuting the conditions independently for each neuron, and store their rotational R² values in `null_shuffle`. Compute the permutation p-value `p_shuffle` as in Week 18.

In [ ]:
r_s = np.random.default_rng(0)
null_shuffle = []
for _ in range(20):
    Y = Xw.copy()
    for n in range(N):
        Y[:, :, n] = ...
    null_shuffle.append(rotation_r2(Y))
p_shuffle = ...
print(f"shuffled: median {np.median(null_shuffle):.3f}, max {max(null_shuffle):.3f};  real {real:.3f};  p = {p_shuffle:.3f}")

In [ ]:
import numpy as np
def test_null():
    "null_shuffle holds 20 rotational R squared values from shuffled data, all far below the real value"
    assert len(null_shuffle) == 20 and max(null_shuffle) < real / 5

def test_p():
    "p_shuffle is (1 + number of nulls at least as large as real) / 21"
    assert abs(p_shuffle - (1 + np.sum(np.array(null_shuffle) >= real)) / 21) < 1e-12

test_null()
test_p()
print("All checks passed")

### Exercise: A covariance-matched surrogate

Compute the temporal covariance `St` (21 × 21, averaged over conditions and neurons) and the neural covariance `Sn` (218 × 218, averaged over conditions and times). Factor them (Cholesky for St, rescaled by N / trace(Sn) so that the total variance comes out right; eigenvectors scaled by the square roots of the clipped eigenvalues for Sn), complete `make_surrogate(r)` and store the rotational R² of 20 surrogates (local rng seed 1) in `null_cov`.

In [ ]:
Tw = Xw.shape[1]
St = np.einsum("ctn,csn->ts", Xw, Xw) / (C * N)
Sn = ...
Lt = np.linalg.cholesky(St * N / np.trace(Sn) + 1e-9 * np.eye(Tw))   # scaled so the total variance matches
w, V = np.linalg.eigh(Sn); Ln = V * np.sqrt(np.clip(w, 0, None))

def make_surrogate(r):
    G = r.normal(size=(C, Tw, N))
    return ...                                    # L_t G_c L_n^T for every condition

r_c = np.random.default_rng(1)
null_cov = [rotation_r2(make_surrogate(r_c)) for _ in range(20)]
print(f"covariance-matched surrogates: median {np.median(null_cov):.3f}, max {max(null_cov):.3f};  real {real:.3f}")

In [ ]:
import numpy as np
def test_cov():
    "the surrogates reproduce the temporal covariance of the data to within 20% on the diagonal"
    Ysur = make_surrogate(np.random.default_rng(9))
    St_s = np.einsum("ctn,csn->ts", Ysur, Ysur) / (C * N)
    assert np.allclose(np.diag(St_s), np.diag(St), rtol=0.2)

def test_null():
    "null_cov holds 20 values, all far below the real rotational R squared"
    assert len(null_cov) == 20 and max(null_cov) < real / 5

test_cov()
test_null()
print("All checks passed")

### Exercise: Timing alone

(a) Build synthetic data: shift each neuron's real condition-averaged response (the mean of Xn over conditions) by a random amount for each condition (uniform within ±40 ms, local rng seed 2, using `shifted`), remove the cross-condition mean, take the window, and store its rotational R² in `r2_shifted`. (b) For each real condition, find the shift in `shifts` whose shifted mean response best matches that condition in the window (least squares), store the shifts in `best_shift`, and compute `timing_r2`, the fraction of the real windowed X explained by the mean-removed timing model.

In [ ]:
mean_resp = Xn.mean(axis=0)                     # times x neurons, the common response

def shifted(s):
    """The common response delayed by s ms (negative s = earlier), by linear interpolation."""
    return np.stack([np.interp(times - s, times, mean_resp[:, n]) for n in range(N)], axis=1)

r_t = np.random.default_rng(2)
Ysyn = np.stack([shifted(s) for s in r_t.uniform(-40, 40, C)])
Ysyn = Ysyn - Ysyn.mean(axis=0, keepdims=True)
r2_shifted = ...

shifts = np.arange(-60, 61, 5)
library = {s: shifted(s) for s in shifts}
best_shift, fit = [], np.zeros_like(Xn)
for c in range(C):
    errs = [np.sum((Xn[c][win] - library[s][win]) ** 2) for s in shifts]
    s = shifts[int(np.argmin(errs))]
    best_shift.append(s); fit[c] = library[s]
model = (fit - fit.mean(axis=0, keepdims=True))[:, win]
timing_r2 = ...
print(f"time-shifted synthetic data: rotational R2 {r2_shifted:.3f} (real {real:.3f})")
print(f"real data: best shifts from {min(best_shift)} to {max(best_shift)} ms;  timing model explains {100 * timing_r2:.1f}% of the condition-specific variance")

In [ ]:
import numpy as np
def test_synthetic():
    "the time-shifted synthetic data give a rotational R squared above the real data's"
    assert r2_shifted > real

def test_fit():
    "best_shift holds one shift per condition from the allowed set, and timing_r2 is the share of the real condition-specific variance the timing model explains"
    assert len(best_shift) == C and set(best_shift) <= set(shifts.tolist())
    assert -0.1 < timing_r2 < 0.2

test_synthetic()
test_fit()
print("All checks passed")